# HMS - Harmful Brain Activity Classification
## -> *Using EfficientNet*

### Importing Libraries


In [1]:
import os
os.environ['KERAS_BACKEND'] = 'torch'

import keras
import keras_cv
import numpy as np
import pandas as pd
import pyarrow
import sklearn
import torch
import tqdm

print(f'keras: {keras.__version__}')
print(f'keras_cv: {keras_cv.__version__}')
print(f'numpy: {np.__version__}')
print(f'pandas: {pd.__version__}')
print(f'pyarrow: {pyarrow.__version__}')
print(f'sklearn: {sklearn.__version__}')
print(f'torch: {torch.__version__}')
print(f'tqdm: {tqdm.__version__}')

2025-05-08 13:42:46.208590: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1746711766.668153      31 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1746711766.809310      31 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


keras: 3.5.0
keras_cv: 0.9.0
numpy: 1.26.4
pandas: 2.2.3
pyarrow: 19.0.1
sklearn: 1.2.2
torch: 2.5.1+cu124
tqdm: 4.67.1


In [2]:
from keras_cv.layers import MixUp, RandomCutout
from keras_cv.models import ImageClassifier
from sklearn.model_selection import StratifiedGroupKFold
from torch.nn.functional import one_hot, pad
from torch.utils.data.dataloader import DataLoader
from tqdm import tqdm

### Defining Paths

In [3]:
DATASET_DIR = '/kaggle/input/hms-harmful-brain-activity-classification'
FOLD = 0
SEED = 2025
SPEC_DIR = '/tmp/dataset/hms-hbac'
os.makedirs(f'{SPEC_DIR}/train_spectrograms/', exist_ok=True)
os.makedirs(f'{SPEC_DIR}/test_spectrograms/', exist_ok=True)

### Loading Parquet and .npy files

In [4]:
# TEST DATA

def load_test_dataset(dataset_dir, spec_dir):
    dataframe = pd.read_csv(f'{dataset_dir}/test.csv')

    dataframe['eeg_path'] = f'{dataset_dir}/test_eegs/' + dataframe.eeg_id.astype('str') + '.parquet'
    dataframe['spec_path'] = f'{dataset_dir}/test_spectrograms/' + dataframe.spectrogram_id.astype('str') + '.parquet'
    dataframe['spec2_path'] = f'{spec_dir}/test_spectrograms/' + dataframe.spectrogram_id.astype('str') + '.npy'
    
    return dataframe

In [5]:
# TRAINING DATA

def load_train_dataset(dataset_dir, spec_dir):
    dataframe = pd.read_csv(f'{dataset_dir}/train.csv')

    dataframe['eeg_path'] = f'{dataset_dir}/train_eegs/' + dataframe.eeg_id.astype('str') + '.parquet'
    dataframe['spec_path'] = f'{dataset_dir}/train_spectrograms/' + dataframe.spectrogram_id.astype('str') + '.parquet'
    dataframe['spec2_path'] = f'{spec_dir}/train_spectrograms/' + dataframe.spectrogram_id.astype('str') + '.npy'  

    name2label = {'Seizure': 0, 'GPD': 1, 'LRDA': 2, 'Other': 3, 'GRDA': 4, 'LPD': 5}

    dataframe['class_name'] = dataframe.expert_consensus.copy()
    dataframe['class_label'] = dataframe.expert_consensus.map(name2label)

    return dataframe

### Converting Spectogram Readings into .npy

In [6]:
def convert_spectrograms(dataset_dir, spec_dir, spectrogram_id, split):
    spec_path = f'{dataset_dir}/{split}_spectrograms/{spectrogram_id}.parquet'
    npy_path = f'{spec_dir}/{split}_spectrograms/{spectrogram_id}.npy'

    spectrogram = pd.read_parquet(spec_path)

    spectrogram = spectrogram.fillna(0)  
    spectrogram = spectrogram.values[:, 1:]
    spectrogram = spectrogram.transpose()  
    spectrogram = spectrogram.astype(np.float32)

    np.save(npy_path, spectrogram)

### Introducing Augmentations

In [7]:
def get_augmenter():
    
    augmenters = [
        MixUp(),
        RandomCutout(height_factor=(1.0, 1.0), width_factor=(0.06, 0.1)),
        RandomCutout(height_factor=(0.06, 0.1), width_factor=(1.0, 1.0))
    ]

    def augment(images, labels):
        data = {'images': images, 'labels': labels}

#         for augmenter in augmenters:
#             if np.random.uniform() < 0.5:
#                 data = augmenter(data, training=True)

        return data['images'], data['labels']

    return augment

In [8]:
def get_decoder(with_labels=True, dtype=32):
    
    def decode_signal(filepath, offset=None):
        
        signal = np.load(filepath)
        signal = torch.from_numpy(signal) 
        
        # Reshaped 400 rows [(frequency bins) , Variable number of columns (time steps)]
        signal = signal.reshape([400, -1]) 
        
        # OFFSET to Extract a time window
        # Library Pad - Padding with zeros on the right if the signal is shorter than 300 time steps.
        
        if offset is not None: 
            offset //= 2
            signal = signal[:, offset:offset+300]

            pad_size = max(0, 300 - signal.size(1))
            signal = pad(signal, (0, pad_size), 'constant', 0)
            signal = signal.reshape([400, 300])

        signal = torch.clip(signal, np.exp(-4.0), np.exp(8))
        signal = torch.log(signal)

        # Apply Normalization
        signal -= torch.mean(signal)
        signal /= torch.std(signal) + 1e-6 # To avoid division by zero

        # EfficientNet expects 3 channel input (RGB)
        signal = torch.tile(signal[..., None], [1, 1, 3])

        return signal

    # One-Hot Encoding
    def decode_label(label):
        return one_hot(torch.tensor(label), num_classes=6) 

    def decode_signal_and_label(filepath, offset, label):
        signal = decode_signal(filepath, offset)
        label = decode_label(label)

        return signal, label

    if with_labels:
        return decode_signal_and_label
    else:
        return decode_signal

### Dataset Creation for Model Training ->

In [9]:
def create_dataset(filepaths, offsets=None, labels=None, augment=True):
    
    augment_fn = get_augmenter()

    decode_fn = get_decoder(labels is not None)

    dataset = []
    for i, file in enumerate(filepaths):
        if labels is not None:
            if offsets is not None:
                dataset.append((file, offsets[i], labels[i]))
            else:
                dataset.append((file, None, labels[i]))
        else:
            if offsets is not None:
                dataset.append((file, offsets[i]))
            else:
                dataset.append((file, None))

    if augment:
        dataset = [augment_fn(*decode_fn(*sample)) for sample in tqdm(dataset)]  
    else:
        dataset = [decode_fn(*sample) for sample in tqdm(dataset)]

    dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

    return dataloader

### Creating .npy files

In [10]:
df = load_train_dataset(DATASET_DIR, SPEC_DIR)
test_df = load_test_dataset(DATASET_DIR, SPEC_DIR)
print('DATAFRAME - \n\n',df.head(2))

spectrogram_ids = df.spectrogram_id.unique()
for id in tqdm(spectrogram_ids):
    convert_spectrograms(DATASET_DIR, SPEC_DIR, id, 'train')

test_spectrogram_ids = test_df.spectrogram_id.unique()
for id in tqdm(test_spectrogram_ids):
    convert_spectrograms(DATASET_DIR, SPEC_DIR, id, 'test')

DATAFRAME - 

        eeg_id  eeg_sub_id  eeg_label_offset_seconds  spectrogram_id  \
0  1628180742           0                       0.0          353733   
1  1628180742           1                       6.0          353733   

   spectrogram_sub_id  spectrogram_label_offset_seconds    label_id  \
0                   0                               0.0   127492639   
1                   1                               6.0  3887563113   

   patient_id expert_consensus  seizure_vote  lpd_vote  gpd_vote  lrda_vote  \
0       42516          Seizure             3         0         0          0   
1       42516          Seizure             3         0         0          0   

   grda_vote  other_vote                                           eeg_path  \
0          0           0  /kaggle/input/hms-harmful-brain-activity-class...   
1          0           0  /kaggle/input/hms-harmful-brain-activity-class...   

                                           spec_path  \
0  /kaggle/input/hms-harm

100%|██████████| 1/1 [00:00<00:00, 24.54it/s]


In [11]:
k_fold = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
df['fold'] = -1
df.reset_index(drop=True, inplace=True)
for fold, (_, valid_set) in enumerate(k_fold.split(df, df.class_label, df.patient_id)):
    df.loc[valid_set, 'fold'] = fold  

sample_df = df.groupby('spectrogram_id')

sample_df = sample_df.head(1) 
sample_df = sample_df.reset_index(drop=True)  
train_df, valid_df = sample_df[sample_df.fold != FOLD], sample_df[sample_df.fold == FOLD]

In [12]:
train_filepaths = train_df.spec2_path.values
train_offsets = train_df.spectrogram_label_offset_seconds.values.astype(int)  # 不转换成int报错.
train_labels = train_df.class_label.values
train_dataset = create_dataset(train_filepaths, train_offsets, train_labels)

test_filepaths = test_df.spec2_path.values
test_dataset = create_dataset(test_filepaths, augment=False)

I0000 00:00:1746712428.715166      31 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13942 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1746712428.715940      31 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13942 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
100%|██████████| 1/1 [00:00<00:00, 452.61it/s]


In [13]:
model = ImageClassifier.from_preset(
    preset='efficientnetv2_b2_imagenet', num_classes=6
)
model.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-4),
              loss=keras.losses.KLDivergence(),
              metrics=[keras.metrics.KLDivergence()])

model.summary()

# Model: "image_classifier"

Model: "image_classifier"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)             │ (None, None, None, 3)       │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ efficient_net_v2b2_backbone          │ (None, None, None, 1408)    │       8,769,374 │
│ (EfficientNetV2Backbone)             │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ avg_pool (GlobalAveragePooling2D)    │ (None, 1408)                │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ predictions (Dense)                  │ (None, 6)                   │           8,454 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 8,777,828 (33.48 MB)

 Trainable params: 8,695,540 (33.17 MB)

 Non-trainable params: 82,288 (321.44 KB)

In [14]:
model.fit(train_dataset,
          epochs=10,
          verbose=1)

Epoch 1/10
274/274 ━━━━━━━━━━━━━━━━━━━━ 222s 793ms/step - kl_divergence: 1.4737 - loss: 1.4737
Epoch 2/10
274/274 ━━━━━━━━━━━━━━━━━━━━ 219s 799ms/step - kl_divergence: 1.0375 - loss: 1.0375
Epoch 3/10
274/274 ━━━━━━━━━━━━━━━━━━━━ 219s 798ms/step - kl_divergence: 0.8957 - loss: 0.8957
Epoch 4/10
274/274 ━━━━━━━━━━━━━━━━━━━━ 218s 797ms/step - kl_divergence: 0.8001 - loss: 0.8001
Epoch 5/10
274/274 ━━━━━━━━━━━━━━━━━━━━ 219s 797ms/step - kl_divergence: 0.7344 - loss: 0.7344
Epoch 6/10
274/274 ━━━━━━━━━━━━━━━━━━━━ 218s 796ms/step - kl_divergence: 0.6816 - loss: 0.6816
Epoch 7/10
274/274 ━━━━━━━━━━━━━━━━━━━━ 218s 796ms/step - kl_divergence: 0.6171 - loss: 0.6171
Epoch 8/10
274/274 ━━━━━━━━━━━━━━━━━━━━ 219s 798ms/step - kl_divergence: 0.5550 - loss: 0.5550
Epoch 9/10
274/274 ━━━━━━━━━━━━━━━━━━━━ 218s 795ms/step - kl_divergence: 0.4889 - loss: 0.4889
Epoch 10/10
274/274 ━━━━━━━━━━━━━━━━━━━━ 218s 796ms/step - kl_divergence: 0.4167 - loss: 0.4167


In [15]:
y_pred = model.predict(test_dataset)
test_df = load_test_dataset(DATASET_DIR, SPEC_DIR)
pred_df = test_df[['eeg_id']].copy()
target_cols = ['seizure_vote', 'lpd_vote', 'gpd_vote', 'lrda_vote', 'grda_vote', 'other_vote']
pred_df[target_cols] = y_pred.tolist()
print(pred_df)
pred_df.to_csv('/kaggle/working/submission.csv', index=False)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 189ms/step
       eeg_id  seizure_vote  lpd_vote  gpd_vote  lrda_vote  grda_vote  \
0  3911565283      0.001377  0.000232  0.969406   0.009834   0.003645   

   other_vote  
0    0.015507  
